## Simulator (fleet of uavs)

In [2]:
from simulator import Simulator
from simulator.config import Color
from simulator.helpers import clean
from simulator.helpers.coordinates import ENUPose, GRAPose
from simulator.planner import AutoPlan, GuidedPlan, Plan
from simulator.visualizer import (
    QGC,
    Gazebo,
    GazMarker,
    NoVisualizer,
    QGCMarker,
    SimVehicle,
)

clean()

## Simulation Positions

In [3]:
num_vehicles = 4

gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241,alt=0,heading=0) 
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading) 

# base_homes= ENUPose.list([  # east, north, up, heading
#     (0., 15., 0., 0.),
#     (15., 0., 0., 0),
#     (5., -20., 0., 30.),
#     (-15., -15., 0., 0),
#     (-15., 0., 0., 45),
# ])

base_homes = []
offset = 15.
for i in range(0, num_vehicles):
  base_homes.append((0., offset, 0., 0.))
  offset += 10.

base_homes = ENUPose.list(base_homes)

## Create Vehicles

In [4]:
sysids = [i for i in range(1, num_vehicles + 1)]
side_len = 5
alt = 5

vehs:list[SimVehicle] = []

for sysid, base_home in zip(sysids, base_homes):
    auto_plan = AutoPlan.square_traj(
        side_len=side_len,
        alt=alt,
        name="simple_auto_plan",
        sysid=sysid,
        gra_origin=gra_origin,
        relative_home=base_home,
    )
    
    guided_plan = GuidedPlan.square_traj(
                                    side_len=side_len, 
                                    alt=alt,
                                    enu_origin=enu_origin,
                                    relative_home=base_home)


    veh = SimVehicle.from_relative(
        sysid=sysid,
        gcs_name=f'Multicolor_{"".join([Color.BLUE.emoji])}',
        plan=guided_plan,
        color=Color.BLUE,
        enu_origin=enu_origin,
        relative_home=base_home,
        relative_path=Plan.create_square_path(side_len=side_len, alt=alt),
        model="iris",
    )
    
    vehs.append(veh)
    
    

## Visualizer

In [5]:
gaz= Gazebo(gra_origin,world_path="simulator/visualizer/gazebo/worlds/runway.world",
            headless=False)
origin_gaz = GazMarker(name="origin",
                    group="origin",
                    pos=enu_origin.unpose(),
                    color=Color.WHITE)
gaz.markers.append(origin_gaz)

## Simulator

In [6]:
simulator = Simulator(
	visualizer=gaz,
	terminals=['gcs'],
	verbose=1,
)
for veh in vehs:
    simulator.add_vehicle(veh)

simulator.show()

## Run

In [7]:
orac = simulator.launch()
orac.run()

00:12:05 - Oracle ⚪ - INFO - 🖥️  Gazebo launched for realistic simulation and 3D visualization.
00:12:05 - Oracle ⚪ - INFO - 🚀 GCS Multicolor_🟦 launched (PID 1112)
00:12:05 - Oracle ⚪ - INFO - 🏁 Starting Oracle with 4 vehicles and 1 GCSs


/home/ubuntu/uav-cyber-sim/simulator/data


# Couldn't connect to accessibility bus: Failed to connect to socket /run/user/1000/at-spi/bus: No such file or directory
# Couldn't connect to accessibility bus: Failed to connect to socket /run/user/1000/at-spi/bus: No such file or directory
# Failed to load module "canberra-gtk-module"
# Failed to load module "canberra-gtk-module"
# Failed to load module "canberra-gtk-module"
# Failed to load module "canberra-gtk-module"
00:13:04 - Oracle ⚪ - INFO - UAV 4 completed mission and exited
00:13:06 - Oracle ⚪ - INFO - UAV 1 completed mission and exited
00:13:07 - Oracle ⚪ - INFO - UAV 2 completed mission and exited
00:13:09 - Oracle ⚪ - INFO - Received DONE from GCS Multicolor_🟦
00:13:09 - Oracle ⚪ - INFO - UAV 3 completed mission and exited
00:13:09 - Oracle ⚪ - INFO - ✅ All GCS threads completed
00:13:09 - Oracle ⚪ - INFO - 🎉 Oracle shutdown complete!


In [8]:
import threading
for thread in threading.enumerate():
    print(f"{thread.name}: alive={thread.is_alive()}, daemon={thread.daemon}")

MainThread: alive=True, daemon=False
IOPub: alive=True, daemon=True
Heartbeat: alive=True, daemon=True
Thread-1 (_watch_pipe_fd): alive=True, daemon=True
Thread-2 (_watch_pipe_fd): alive=True, daemon=True
Control: alive=True, daemon=True
Shell channel: alive=True, daemon=True
IPythonHistorySavingThread: alive=True, daemon=True
Thread-4: alive=True, daemon=True
